### 종합 실습 과제: 최적의 하이퍼파라미터 조합을 찾아라!

이제 이론과 기법들을 총동원하여 모델의 성능을 극한까지 끌어올릴 시간입니다.

`과제 목표`: Day 2-Part 1에서 만든 `AdvancedClassifier` 모델의 테스트 정확도를 하이퍼파라미터 튜닝을 통해 `최대한 높여보세요.`

`요구사항:`

1.  아래에 정의된 하이퍼파라미터 탐색 공간(`param_grid`) 내에서 `그리드 탐색(Grid Search)`을 수행하는 코드를 완성하세요.
2.  각 조합에 대해 모델을 훈련하고 `테스트 데이터셋에 대한 정확도`를 측정 및 기록하세요.
3.  모든 조합의 테스트가 끝나면, 가장 높은 정확도를 보인 `최고의 하이퍼파라미터 조합`과 그때의 `테스트 정확도`를 출력하세요.
4.  (도전 과제) 결과를 보기 쉽게 DataFrame으로 정리하고, 정확도를 기준으로 내림차순 정렬하여 상위 5개 조합을 출력해보세요.

아래의 Starter Code를 바탕으로 과제를 완성해 보세요!

In [2]:
# [기본 라이브러리 및 데이터 준비 코드]
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import itertools

# 0. 데이터 준비
X, y = load_breast_cancer(return_X_y=True)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_dataset = BreastCancerDataset(X_train, y_train)
val_dataset = BreastCancerDataset(X_val, y_val)
test_dataset = BreastCancerDataset(X_test, y_test)

input_features = X_train.shape[1]
num_classes = 2

In [3]:
# [과제용 Starter Code]

# 1. 하이퍼파라미터 탐색 공간 정의
param_grid = {
    'lr': [0.01, 0.001, 0.0001],
    'batch_size': [16, 32, 64],
    'optimizer': ['Adam', 'SGD'],
    'dropout_p': [0.3, 0.5]
}

# 2. 모델 정의 (AdvancedClassifier 재사용)
class AdvancedClassifier(nn.Module):
    def __init__(self, num_features, num_classes, dropout_p=0.4):
        super(AdvancedClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(num_features, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p=dropout_p),
            nn.Linear(32, num_classes)
        )
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

    def forward(self, x):
        return self.net(x)

In [4]:
results = []
best_accuracy = 0.0
best_params = {}

In [5]:
# 3. 그리드 탐색 루프 구현
# HINT: itertools.product를 사용하면 모든 조합을 쉽게 생성할 수 있습니다.
grid = list(itertools.product(*param_grid.values()))
print(f"총 {len(grid)}개의 하이퍼파라미터 조합을 테스트합니다.")

for i, params in enumerate(grid):
    # 딕셔너리 형태로 파라미터 재구성
    p = dict(zip(param_grid.keys(), params))
    print(f"\n--- {i+1}/{len(grid)} 번째 조합 테스트: {p} ---")
    
    # ====================== 과제 영역 시작 ======================
    # TODO 1: 하이퍼파라미터에 따라 DataLoader와 모델, 옵티마이저를 설정하세요.
    train_loader = DataLoader(train_dataset, batch_size=p['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=p['batch_size'])
    test_loader = DataLoader(test_dataset, batch_size=p['batch_size'])
    
    model = AdvancedClassifier(input_features, num_classes, dropout_p=p['dropout_p'])
    criterion = nn.CrossEntropyLoss()
    
    if p['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=p['lr'])
    else: # SGD
        optimizer = optim.SGD(model.parameters(), lr=p['lr'], momentum=0.9)

    # TODO 2: 모델 학습 루프를 작성하세요. (약 50 에포크)
    # 이 루프는 각 조합마다 반복 실행됩니다.
    num_epochs = 50
    for epoch in range(num_epochs):
        model.train()
        for features, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
    # TODO 3: 학습이 끝난 모델로 테스트 데이터셋의 정확도를 계산하세요.
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in test_loader:
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    current_accuracy = 100 * correct / total
    print(f"테스트 정확도: {current_accuracy:.2f}%")
    
    # TODO 4: 결과를 기록하고, 최고 성능 모델인지 확인 및 업데이트하세요.
    results.append({**p, 'accuracy': current_accuracy})
    
    if current_accuracy > best_accuracy:
        best_accuracy = current_accuracy
        best_params = p
        # (선택) 최고 성능 모델 저장
        # torch.save(model.state_dict(), 'best_model.pth')
        
    # ====================== 과제 영역 종료 ======================

# 4. 최종 결과 출력
print("\n================ 최종 결과 ================")
print(f"최고 정확도: {best_accuracy:.2f}%")
print(f"최적 하이퍼파라미터: {best_params}")

# 5. (도전 과제) 결과를 DataFrame으로 출력
results_df = pd.DataFrame(results)
print("\n--- 상위 5개 결과 ---")
print(results_df.sort_values(by='accuracy', ascending=False).head(5))

총 36개의 하이퍼파라미터 조합을 테스트합니다.

--- 1/36 번째 조합 테스트: {'lr': 0.01, 'batch_size': 16, 'optimizer': 'Adam', 'dropout_p': 0.3} ---
테스트 정확도: 96.49%

--- 2/36 번째 조합 테스트: {'lr': 0.01, 'batch_size': 16, 'optimizer': 'Adam', 'dropout_p': 0.5} ---
테스트 정확도: 95.61%

--- 3/36 번째 조합 테스트: {'lr': 0.01, 'batch_size': 16, 'optimizer': 'SGD', 'dropout_p': 0.3} ---
테스트 정확도: 94.74%

--- 4/36 번째 조합 테스트: {'lr': 0.01, 'batch_size': 16, 'optimizer': 'SGD', 'dropout_p': 0.5} ---
테스트 정확도: 97.37%

--- 5/36 번째 조합 테스트: {'lr': 0.01, 'batch_size': 32, 'optimizer': 'Adam', 'dropout_p': 0.3} ---
테스트 정확도: 93.86%

--- 6/36 번째 조합 테스트: {'lr': 0.01, 'batch_size': 32, 'optimizer': 'Adam', 'dropout_p': 0.5} ---
테스트 정확도: 94.74%

--- 7/36 번째 조합 테스트: {'lr': 0.01, 'batch_size': 32, 'optimizer': 'SGD', 'dropout_p': 0.3} ---
테스트 정확도: 95.61%

--- 8/36 번째 조합 테스트: {'lr': 0.01, 'batch_size': 32, 'optimizer': 'SGD', 'dropout_p': 0.5} ---
테스트 정확도: 96.49%

--- 9/36 번째 조합 테스트: {'lr': 0.01, 'batch_size': 64, 'optimizer': 'Adam', 'dropout_p': 0.3

In [7]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def objective(trial):
    # 하이퍼파라미터 정의
    hidden_size = trial.suggest_int('hidden_size', 32, 256)
    num_layers = trial.suggest_int('num_layers', 1, 4)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    
    # 모델 생성 - AdvancedClassifier의 실제 파라미터에 맞춰 수정
    model = AdvancedClassifier(
        num_features=X_train.shape[1],
        num_classes=len(np.unique(y_train)),
        dropout_p=dropout_rate
    )
    
    # 데이터로더 생성
    train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # 옵티마이저와 손실 함수
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    
    # 학습 루프 (간단한 버전으로 빠른 평가)
    num_epochs = 20  # 빠른 평가를 위해 에포크 수 줄임
    model.train()
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        for features, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
    
    # 검증 성능 평가
    model.eval()
    test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in test_loader:
            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    return accuracy

# Optuna를 사용한 베이지안 최적화
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# 최적 하이퍼파라미터 출력
print("최적 하이퍼파라미터:")
print(study.best_params)
print(f"최고 정확도: {study.best_value:.2f}%")

[I 2025-06-24 12:18:41,616] A new study created in memory with name: no-name-44c4bf06-ed97-4094-bdff-2f84260b9bd6
[I 2025-06-24 12:18:42,469] Trial 0 finished with value: 96.49122807017544 and parameters: {'hidden_size': 153, 'num_layers': 3, 'learning_rate': 0.004674198309358627, 'batch_size': 16, 'dropout_rate': 0.39629197337627275}. Best is trial 0 with value: 96.49122807017544.
[I 2025-06-24 12:18:43,303] Trial 1 finished with value: 97.36842105263158 and parameters: {'hidden_size': 197, 'num_layers': 2, 'learning_rate': 0.0006673289485161328, 'batch_size': 16, 'dropout_rate': 0.2992202302590984}. Best is trial 1 with value: 97.36842105263158.
[I 2025-06-24 12:18:43,525] Trial 2 finished with value: 92.10526315789474 and parameters: {'hidden_size': 225, 'num_layers': 3, 'learning_rate': 0.00013600172206285586, 'batch_size': 64, 'dropout_rate': 0.3056433224095144}. Best is trial 1 with value: 97.36842105263158.
[I 2025-06-24 12:18:43,763] Trial 3 finished with value: 90.350877192982

최적 하이퍼파라미터:
{'hidden_size': 103, 'num_layers': 2, 'learning_rate': 0.00901042745778621, 'batch_size': 16, 'dropout_rate': 0.1858822171667628}
최고 정확도: 98.25%


In [8]:
# 최적화 과정 시각화
trials = study.trials
values = [trial.value for trial in trials]
params_history = []
for trial in trials:
    params_history.append(list(trial.params.values()))

# 서브플롯 생성
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('최적화 과정', '하이퍼파라미터 변화'),
    vertical_spacing=0.1
)

# 정확도 변화
fig.add_trace(
    go.Scatter(
        x=list(range(1, len(values) + 1)),
        y=values,
        mode='lines+markers',
        name='정확도',
        line=dict(color='blue')
    ),
    row=1, col=1
)

# 최고 정확도 추적
best_so_far = np.maximum.accumulate(values)
fig.add_trace(
    go.Scatter(
        x=list(range(1, len(best_so_far) + 1)),
        y=best_so_far,
        mode='lines',
        name='최고 정확도',
        line=dict(color='red', dash='dash')
    ),
    row=1, col=1
)

# 하이퍼파라미터 변화
param_names = list(study.best_params.keys())
colors = ['green', 'orange', 'purple', 'brown', 'pink']
for i, param_name in enumerate(param_names):
    param_values = [params[i] for params in params_history]
    fig.add_trace(
        go.Scatter(
            x=list(range(1, len(param_values) + 1)),
            y=param_values,
            mode='lines+markers',
            name=param_name,
            line=dict(color=colors[i % len(colors)])
        ),
        row=2, col=1
    )

fig.update_layout(
    title='AdvancedClassifier 베이지안 최적화 결과',
    height=600,
    showlegend=True
)

fig.update_xaxes(title_text='시도 횟수', row=2, col=1)
fig.update_yaxes(title_text='정확도 (%)', row=1, col=1)
fig.update_yaxes(title_text='하이퍼파라미터 값', row=2, col=1)

fig.show()

In [9]:
# 중요도 분석
importance = optuna.importance.get_param_importances(study)
print("\n하이퍼파라미터 중요도:")
for param, imp in importance.items():
    print(f"{param}: {imp:.4f}")


하이퍼파라미터 중요도:
dropout_rate: 0.4894
hidden_size: 0.2871
batch_size: 0.1326
learning_rate: 0.0746
num_layers: 0.0163


In [14]:
# 최적 하이퍼파라미터로 최종 모델 학습
best_model = AdvancedClassifier(
    num_features=X_train.shape[1],
    num_classes=len(np.unique(y_train)),
    dropout_p=study.best_params['dropout_rate']
)

best_optimizer = optim.Adam(best_model.parameters(), lr=study.best_params['learning_rate'])
best_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
best_loader = DataLoader(best_dataset, batch_size=study.best_params['batch_size'], shuffle=True)

In [15]:
# 최종 학습 (더 많은 에포크)
num_epochs = 100
best_model.train()
for epoch in range(num_epochs):
    for features, labels in best_loader:
        best_optimizer.zero_grad()
        outputs = best_model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        best_optimizer.step()

In [16]:
# 최종 성능 평가
best_model.eval()
test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))
test_loader = DataLoader(test_dataset, batch_size=study.best_params['batch_size'], shuffle=False)

correct = 0
total = 0
with torch.no_grad():
    for features, labels in test_loader:
        outputs = best_model(features)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_accuracy = 100 * correct / total
print(f"최종 모델 정확도: {final_accuracy:.2f}%")


최종 모델 정확도: 95.61%
